# Tamil Nadu 9BA hourly peak-week unit commitment

This notebook loads the independent nine-balancing-area FY2025-26 model, selects the Monday-Sunday week containing the statewide annual demand peak, solves unit commitment with HiGHS, checks nodal balances and transmission loading, and exports the solved network.

No equipment faults, forced outages, contingencies, or short-circuit events are introduced.

In [1]:
from pathlib import Path

import pandas as pd
import pypsa

ROOT = Path.cwd()
if not (ROOT / "9_BA" / "model").exists():
    ROOT = ROOT.parent

INPUT_DIR = ROOT / "9_BA" / "model"
RESULT_FILE = ROOT / "9_BA" / "tamil_nadu_9ba_2025_26.nc"

n_year = pypsa.Network(INPUT_DIR)

assert len(n_year.buses) == 9, f"Expected 9 buses, found {len(n_year.buses)}"
assert len(n_year.links) == 22, f"Expected 22 corridors, found {len(n_year.links)}"
assert len(n_year.snapshots) == 8760
assert n_year.generators.committable.any(), "No committable generators found"
assert (n_year.snapshots.to_series().diff().dropna() == pd.Timedelta(hours=1)).all()

annual_demand = n_year.loads_t.p_set.sum(axis=1)
peak_timestamp = annual_demand.idxmax()
week_start = peak_timestamp.normalize() - pd.Timedelta(days=peak_timestamp.weekday())
week_end = week_start + pd.Timedelta(days=7)
week = n_year.snapshots[
    (n_year.snapshots >= week_start) & (n_year.snapshots < week_end)
]
assert len(week) == 168, f"Expected 168 hourly snapshots, found {len(week)}"

n = n_year.copy(snapshots=week)

OIL_GAS_BUDGET_CONSTRAINT = "observed_fy2025_26_oil_gas_generation_target"
oil_gas_budget = (
    pd.read_csv(INPUT_DIR / "oil_gas_daily_energy_budget.csv", parse_dates=["date"])
    .set_index("date")
)
week_dates = pd.DatetimeIndex(week).normalize().unique()
week_oil_gas_target_mwh = float(
    oil_gas_budget.loc[week_dates, "daily_energy_target_mwh"].sum()
)
assert OIL_GAS_BUDGET_CONSTRAINT in n.global_constraints.index
n.global_constraints.at[
    OIL_GAS_BUDGET_CONSTRAINT, "constant"
] = week_oil_gas_target_mwh

def add_daily_oil_gas_targets(network, snapshots):
    gas_units = network.generators.index[network.generators.carrier == "oil_gas"]
    dispatch = network.model["Generator-p"]
    normalized = pd.DatetimeIndex(snapshots).normalize()
    for day in normalized.unique():
        day_snapshots = snapshots[normalized == day]
        daily_target_mwh = float(oil_gas_budget.at[day, "daily_energy_target_mwh"])
        daily_dispatch_mwh = dispatch.sel(
            name=gas_units, snapshot=day_snapshots
        ).sum()
        network.model.add_constraints(
            daily_dispatch_mwh,
            "==",
            daily_target_mwh,
            name=f"OilGasDailyTarget-{day:%Y%m%d}",
        )

print(f"Loaded {len(n_year.buses)} balancing areas and {len(n_year.links)} corridors")
print(f"Committable generators: {int(n_year.generators.committable.sum())}")
print(f"Annual peak: {annual_demand.loc[peak_timestamp]:,.2f} MW at {peak_timestamp}")
print(f"Optimizing: {week_start} to {week_end - pd.Timedelta(hours=1)}")
print(f"Oil-and-gas target for selected week: {week_oil_gas_target_mwh / 1_000.0:,.2f} MU")

INFO:pypsa.network.io:New version 1.3.0 available! (Current: 1.2.4)


INFO:pypsa.network.io:Imported network 'Tamil Nadu FY 2025-26 nine-balancing-area hourly UC inputs with July 2026 capacity overlay' has buses, carriers, generators, global_constraints, links, loads, storage_units


Loaded 9 balancing areas and 22 corridors
Committable generators: 166
Annual peak: 19,987.33 MW at 2025-07-11 16:00:00
Optimizing: 2025-07-07 00:00:00 to 2025-07-13 23:00:00
Oil-and-gas cap for selected week: 25.12 MU


In [2]:
n

PyPSA Network 'Tamil Nadu FY 2025-26 nine-balancing-area hourly UC inputs with July 2026 capacity overlay'
----------------------------------------------------------------------------------------------------------
Components:
 - Bus: 9
 - Carrier: 13
 - Generator: 407
 - GlobalConstraint: 1
 - Link: 22
 - Load: 9
 - StorageUnit: 4
Snapshots: 168

## Model overview

Installed generator and storage capacities are shown by balancing area and carrier. Transmission limits are fixed and bidirectional. These tables describe the intact network; no fault-related availability modifications are made.

In [3]:
generator_capacity = (
    n.generators.groupby(["bus", "carrier"]).p_nom.sum()
    .rename("capacity_MW")
    .to_frame()
)
storage_capacity = (
    n.storage_units.groupby(["bus", "carrier"]).p_nom.sum()
    .rename("capacity_MW")
    .to_frame()
)
corridors = n.links[
    ["bus0", "bus1", "p_nom", "p_min_pu", "efficiency"]
].rename(columns={"p_nom": "capacity_MW"})

display(generator_capacity)
display(storage_capacity)
display(corridors)

capacity_MW
bus           carrier                      
Chennai North bio_power            5.735511
              coal              4677.500000
              market_import      570.033536
              oil_gas            120.000000
              solar              340.296187
...                                     ...
Villupuram    market_import      685.248795
              small_hydro          7.500000
              solar              557.778356
              unserved_energy  10186.543702
              wind                 1.716869

[65 rows x 1 columns]

,,capacity_MW
bus,carrier,
Coimbatore,hydro,400.0


,bus0,bus1,capacity_MW,p_min_pu,efficiency
name,,,,,
chennai_north__chennai_south,Chennai North,Chennai South,3500.0,-1.0,1.0
chennai_north__vellore,Chennai North,Vellore,1000.0,-1.0,1.0
chennai_north__villupuram,Chennai North,Villupuram,2000.0,-1.0,1.0
chennai_north__trichy,Chennai North,Trichy,9000.0,-1.0,1.0
chennai_south__vellore,Chennai South,Vellore,1000.0,-1.0,1.0
chennai_south__villupuram,Chennai South,Villupuram,1000.0,-1.0,1.0
chennai_south__trichy,Chennai South,Trichy,1000.0,-1.0,1.0
vellore__villupuram,Vellore,Villupuram,3500.0,-1.0,1.0
vellore__erode,Vellore,Erode,6750.0,-1.0,1.0


## Solve the peak week

This is a mixed-integer unit-commitment optimization. Full-year inputs remain available in the CSV model, but only the 168-hour peak week is solved here.

In [4]:
status, termination_condition = n.optimize(
    solver_name="highs",
    solver_options={"mip_rel_gap": 1e-3, "log_to_console": False},
    extra_functionality=add_daily_oil_gas_targets,
    include_objective_constant=False,
)

if termination_condition != "optimal":
    raise RuntimeError(
        f"Optimization failed: {status}, {termination_condition}"
    )

print(f"Optimization: {status} ({termination_condition})")

       'mettur_tps__unit_01', 'mettur_tps__unit_02', 'mettur_tps__unit_03',
       'mettur_tps__unit_04', 'mettur_tps__unit_05', 'muthiara_tpp__unit_01',
       'muthiara_tpp__unit_02', 'neyveli_ext_tps_i__unit_01',
       ...
       'additional_2026_07__diesel_samayanallur__unit_05',
       'additional_2026_07__diesel_samayanallur__unit_06',
       'additional_2026_07__diesel_samayanallur__unit_07',
       'additional_2026_07__diesel_samalpatti__unit_01',
       'additional_2026_07__diesel_samalpatti__unit_02',
       'additional_2026_07__diesel_samalpatti__unit_03',
       'additional_2026_07__diesel_samalpatti__unit_04',
       'additional_2026_07__diesel_samalpatti__unit_05',
       'additional_2026_07__diesel_samalpatti__unit_06',
       'additional_2026_07__diesel_samalpatti__unit_07'],
      dtype='object', name='name', length=166).


INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.model:Solver options:
 - mip_rel_gap: 0.001
 - log_to_console: False


INFO:linopy.io:Writing objective.


Writing constraints.:   0%|          | 0/34 [00:00<?, ?it/s]

...

Writing binary variables.: 100%|██████████| 3/3 [00:00<00:00, 251.24it/s]


INFO:linopy.io: Writing time: 0.67s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 157752 primals, 509966 duals
Objective: 5.52e+09
Solver: highs
Runtime: 309.49s
MIP gap: 6.20e-05
Dual bound: 5.52e+09
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-status-p-fixed-upper, Generator-start_up-p-fixed-upper, Generator-shut_down-p-fixed-upper, Generator-fix-p-lower, Generator-fix-p-upper, Generator-com-p-lower, Generator-com-p-upper, Generator-com-transition-start-up, Generator-com-transition-shut-down, Generator-com-up-time, Generator-com-down-time, Generator-com-status-min_down_time_must_stay_up, Generator-p-ramp_limit_up, Generator-p-ramp_limit_down, Link-fix-p-lower, Link-fix-p-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-energy_balance, OilGasDailyCap-20250707, OilGasDailyCap-20250708, OilGasDailyCap-20250709, OilGasDailyCap-20250710, OilGasDailyCap-20250711, OilGasDailyCap-20250712, OilGasDailyCap-20250713 were not assigned to the network.


Optimization: ok (optimal)


In [5]:
weights = n.snapshot_weightings.generators
unserved_units = n.generators.index[
    n.generators.carrier == "unserved_energy"
]
unserved_mwh = (
    n.generators_t.p[unserved_units]
    .mul(weights, axis=0)
    .sum()
    .sum()
)
gas_units = n.generators.index[n.generators.carrier == "oil_gas"]
gas_daily_mwh = (
    n.generators_t.p[gas_units].sum(axis=1).mul(weights).groupby(n.snapshots.normalize()).sum()
)
gas_daily_targets_mwh = oil_gas_budget.loc[gas_daily_mwh.index, "daily_energy_target_mwh"]
max_gas_target_deviation_mwh = float((gas_daily_mwh - gas_daily_targets_mwh).abs().max())
assert max_gas_target_deviation_mwh < 1e-5

nodal_balance_residual_mw = n.buses_t.p.sum(axis=1).abs().max()
link_loading = n.links_t.p0.abs().div(n.links.p_nom, axis=1)
max_link_loading = link_loading.max().max()
most_loaded_link = link_loading.max().idxmax()
most_loaded_time = link_loading[most_loaded_link].idxmax()

print(f"Objective: {n.objective:,.2f}")
print(f"Unserved energy: {unserved_mwh:,.6f} MWh")
print(f"Oil-and-gas generation: {gas_daily_mwh.sum() / 1_000.0:,.2f} MU")
print(f"Maximum daily oil-and-gas target deviation: {max_gas_target_deviation_mwh:.3e} MWh")
print(f"Maximum nodal-balance residual: {nodal_balance_residual_mw:,.3e} MW")
print(
    f"Maximum corridor loading: {100 * max_link_loading:.2f}% "
    f"on {most_loaded_link} at {most_loaded_time}"
)

assert nodal_balance_residual_mw < 1e-5
assert max_link_loading <= 1.0 + 1e-6

n.export_to_netcdf(RESULT_FILE)
print(f"Exported solved network to {RESULT_FILE}")

Objective: 5,517,214,507.90
Unserved energy: 0.000000 MWh
Oil-and-gas generation: 0.00 MU
Maximum daily oil-and-gas cap utilization: 0.00%
Maximum nodal-balance residual: 8.713e-10 MW
Maximum corridor loading: 100.00% on villupuram__tirunelveli at 2025-07-10 06:00:00


INFO:pypsa.network.io:Exported network 'Tamil Nadu FY 2025-26 nine-balancing-area hourly UC inputs with July 2026 capacity overlay' saved to 'C:\Users\b076218\Documents\Filkassen\DEA_GE_github\PyPSA-TamilNadu\9_BA\tamil_nadu_9ba_2025_26.nc contains: loads, storage_units, carriers, global_constraints, buses, sub_networks, generators, links


Exported solved network to C:\Users\b076218\Documents\Filkassen\DEA_GE_github\PyPSA-TamilNadu\9_BA\tamil_nadu_9ba_2025_26.nc
